In [1]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

In [3]:
dir_base = Path("tiles/data/tiles-v2-2021-2024")
dir_so2 = Path("tiles/data/tiles-v2-2021-2024")
dir_out = Path("tiles/data/tiles-v3-so2-p99-2021-2024")

print("Cargando dataset base v2...")
df_base = pd.read_parquet(dir_base / "tiles_meta.parquet")
with np.load(dir_base / "tiles_train.npz") as data:
    images_base = data["data"]
    bands = data["bands"]

print(f"Metadata base: {df_base.shape[0]} registros")
print(f"Imágenes base: {images_base.shape}")
print(f"Clases base:\n{df_base['clase'].value_counts()}")

print("\nQuitando SO2 anterior...")
mask_keep = df_base["clase"] != "contaminacion_alta_SO2"
df_keep = df_base[mask_keep].copy()
images_keep = images_base[mask_keep.values]

print(f"Metadata sin SO2: {df_keep.shape[0]} registros")
print(f"Imágenes sin SO2: {images_keep.shape}")

print("\nCargando SO2 p99...")
df_so2 = pd.read_parquet(dir_so2 / "tiles_meta_so2_p99.parquet")
with np.load(dir_so2 / "tiles_train_so2_p99.npz") as data:
    images_so2 = data["data"]
    bands_so2 = data["bands"]

df_so2 = df_so2.copy().head(230)
images_so2 = images_so2[:230]
df_so2["clase"] = "contaminacion_alta_SO2"

assert len(df_so2) == 230, "SO2 p99 no tiene 230 registros"
assert images_so2.shape == (230, 13, 64, 64), "Tensor SO2 p99 inesperado"
assert list(bands) == list(bands_so2), "Bandas no coinciden"

print(f"Metadata SO2 p99: {df_so2.shape[0]} registros")
print(f"Imágenes SO2 p99: {images_so2.shape}")
print(df_so2[["so2", "ndvi", "ndbi", "scl_pct"]].describe().round(6))

print("\nConsolidando v3...")
df_final = pd.concat([df_keep, df_so2], ignore_index=True)
images_final = np.concatenate([images_keep, images_so2], axis=0).astype("float32")

orden = [
    "contaminacion_alta_NO2",
    "contaminacion_alta_SO2",
    "ozono_anomalo",
    "suelo_urbano",
    "vegetacion_densa",
]
conteo = df_final["clase"].value_counts().reindex(orden)

print("\n=== CONSOLIDACIÓN EXITOSA ===")
print(f"Total muestras finales: {df_final.shape[0]}")
print(f"Tensor final: {images_final.shape}")
print("\nConteo final por clase:")
print(conteo)

assert df_final.shape[0] == 1150, "Total final inesperado"
assert images_final.shape == (1150, 13, 64, 64), "Tensor final inesperado"
assert conteo.min() == 230 and conteo.max() == 230, "Dataset final no está balanceado"

print("\nGuardando v3 sin sobrescribir v2...")
if dir_out.exists():
    shutil.rmtree(dir_out)
dir_out.mkdir(parents=True, exist_ok=True)

df_final.to_parquet(dir_out / "tiles_meta.parquet", index=False)
np.savez_compressed(dir_out / "tiles_train.npz", data=images_final, bands=bands)

df_so2.to_parquet(dir_out / "tiles_meta_so2_p99_reemplazo.parquet", index=False)
np.savez_compressed(dir_out / "tiles_train_so2_p99_reemplazo.npz", data=images_so2, bands=bands)

print(f"Listo: {dir_out}")


Cargando dataset base v2...
Metadata base: 1150 registros
Imágenes base: (1150, 13, 64, 64)
Clases base:
clase
contaminacion_alta_NO2    230
vegetacion_densa          230
ozono_anomalo             230
contaminacion_alta_SO2    230
suelo_urbano              230
Name: count, dtype: int64

Quitando SO2 anterior...
Metadata sin SO2: 920 registros
Imágenes sin SO2: (920, 13, 64, 64)

Cargando SO2 p99...
Metadata SO2 p99: 230 registros
Imágenes SO2 p99: (230, 13, 64, 64)
              so2        ndvi        ndbi     scl_pct
count  230.000000  230.000000  230.000000  230.000000
mean     0.000431    0.545451   -0.156686    0.893563
std      0.000232    0.173046    0.138236    0.180217
min      0.000194    0.073494   -0.446711    0.321289
25%      0.000264    0.420880   -0.264451    0.870056
50%      0.000360    0.580043   -0.175025    0.999023
75%      0.000515    0.679019   -0.049317    1.000000
max      0.001800    0.847045    0.198340    1.000000

Consolidando v3...

=== CONSOLIDACIÓN EXITO